# Case-based breast image classification (simulation)

Fit a whitening transform on training cases. Classify each query by the difference in mean distance to its five nearest cases from each class, with logistic calibration on a separate subset. Compare all pixels with a version that masks the top-left corner. Add a label-correlated corner artifact in training, then evaluate aligned, clean and reversed conditions. The nearest-case mechanism follows the rationale of the mammography paper and uses whole-image descriptors. Generate independent lesion phantoms with shape and texture cues. The corner artifact is generated separately from the lesion.

In [ ]:
from pathlib import Path
import subprocess, sys
root = Path.cwd() / 'case-based-breast-imaging-heavy-run'
root.mkdir(exist_ok=True)
subprocess.run([sys.executable,'-m','pip','install',*['numpy==2.2.6', 'scipy==1.15.3', 'scikit-learn==1.7.2', 'joblib==1.6.0']],check=True)

In [ ]:
(root / 'train.py').write_text('import json\nimport os\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import (\n    average_precision_score,\n    roc_auc_score,\n)\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\n\nROOT = Path(__file__).resolve().parent\nDATA = Path(os.environ.get("PORTFOLIO_DATA", ROOT / "data"))\nSEEDS = [13, 37, 61]\n\n\ndef save(obj, name):\n    folder = ROOT / "checkpoints"\n    folder.mkdir(exist_ok=True)\n    joblib.dump(obj, folder / (name + ".joblib"), compress=3)\n\n\ndef scores(y, p):\n    return {\n        "auroc": float(roc_auc_score(y, p)),\n        "average_precision": float(average_precision_score(y, p)),\n    }\n\n\ndef dataset(seed):\n    rng = np.random.default_rng(seed)\n    u, v = np.mgrid[-1:1:28j, -1:1:28j]\n\n    def sample(n):\n        y = rng.integers(2, size=n)\n        x = []\n        for k in y:\n            a, b = rng.normal(0, 0.15, 2)\n            r = ((u - a) / (0.29 + 0.05 * k)) ** 2 + ((v - b) / 0.31) ** 2\n            pattern = np.exp(-r) * (1 + 0.2 * k * np.cos(u * 25))\n            x.append(\n                np.clip(\n                    0.42\n                    + pattern * rng.uniform(0.15, 0.44)\n                    + rng.normal(0, 0.13, u.shape),\n                    0,\n                    1,\n                )\n            )\n        return np.asarray(x), y\n\n    return (*sample(1200), *sample(440))\n\n\nfrom scipy.ndimage import gaussian_filter\nfrom sklearn.decomposition import PCA\nfrom sklearn.metrics import pairwise_distances\nfrom sklearn.model_selection import train_test_split\n\n\ndef descriptors(x):\n    blurred = gaussian_filter(x, (0, 1, 1))\n    gradient = np.abs(x - blurred)\n    return np.column_stack(\n        [\n            a.reshape(len(a), 7, 4, 7, 4).mean((2, 4)).reshape(len(a), -1)\n            for a in (x, gradient)\n        ]\n    )\n\n\ndef run(seed):\n    x, y, z, t = dataset(seed)\n    np.random.default_rng(seed)\n\n    def mark(images, labels):\n        a = images.copy()\n        a[:, :5, :5] += np.where(labels[:, None, None] == 1, 0.46, -0.46)\n        return np.clip(a, 0, 1)\n\n    train = mark(x, y)\n    test = {"aligned": mark(z, t), "reversed": mark(z, 1 - t), "clean": z}\n    ids = np.arange(len(y))\n    calibration, bank = train_test_split(\n        ids, test_size=0.78, stratify=y, random_state=seed\n    )\n    output = {}\n    for method in ("full", "center"):\n\n        def features(img):\n            if method == "center":\n                img = img.copy()\n                img[:, :8, :8] = 0.5\n            return descriptors(img)\n\n        enc = make_pipeline(StandardScaler(), PCA(20, whiten=True, random_state=seed))\n        a = enc.fit_transform(features(train[bank]))\n\n        def score(b):\n            ds = pairwise_distances(b, a, squared=True)\n            neg = ds[:, y[bank] == 0]\n            pos = ds[:, y[bank] == 1]\n            return np.partition(neg, 4, axis=1)[:, :5].mean(1) - np.partition(\n                pos, 4, axis=1\n            )[:, :5].mean(1)\n\n        lr = LogisticRegression().fit(\n            score(enc.transform(features(train[calibration])))[:, None], y[calibration]\n        )\n        for shift, images in test.items():\n            b = enc.transform(features(images))\n            p = lr.predict_proba(score(b)[:, None])[:, 1]\n            output[f"{method}_{shift}"] = scores(t, p)\n        save((enc, a, y[bank], lr), f"{method}-{seed}")\n    return output\n\n\nif __name__ == "__main__":\n    runs = {s: run(s) for s in SEEDS}\n    first = next(iter(runs.values()))\n    means = {\n        k: {m: float(np.mean([r[k][m] for r in runs.values()])) for m in v}\n        for k, v in first.items()\n    }\n    (ROOT / "results.json").write_text(\n        json.dumps({"seeds": SEEDS, "runs": runs, "means": means}, indent=2)\n    )\n    print(means)\n',encoding='utf8')
subprocess.run([sys.executable,'train.py'],cwd=root,check=True)

In [ ]:
print((root/'results.json').read_text())